# x86-64 Assembly basics

> **Note:** the notebooks in `examples/` are reset every time your server starts, so they are always up to date. To keep your changes, copy this notebook into `work/` (right-click it in the file browser → *Copy*, then paste it in `work/`).

This notebook uses the **x86-64 Assembly (NASM)** kernel. Your code runs on an **emulated x86-64 CPU** that keeps its state
between cells: after each cell you see all the registers, and the ones that changed are **highlighted**.
Run the cells in order with **Shift + Enter**, and watch the table.

We use **NASM** syntax (Intel style): `instruction destination, source`.

| Kernel command | What it does |
|---|---|
| `%regs` | show the registers |
| `%mem <address> [length]` | show memory, e.g. `%mem DATA 16` or `%mem rsp 16` |
| `%reset` | start again with a fresh CPU |
| `%%program` | run a complete program (sections, `_start`, system calls) |
| `%help` | show help |

## 1. Registers

The CPU has 16 general-purpose 64-bit registers: `rax rbx rcx rdx rsi rdi rbp rsp r8 … r15`.
`rip` points to the next instruction, and `rflags` holds the **flags** (results of comparisons).

Let's start with a fresh CPU. Every register is 0 except `rsp`, the stack pointer.

In [ ]:
%reset

## 2. `mov`: copying values

`mov dest, src` copies `src` into `dest`. The source can be a number (an *immediate*), another register or memory.
Numbers can be decimal, hexadecimal (`0xff`), binary (`0b1010`) or a character (`'A'`).

In [ ]:
mov rax, 42
mov rbx, 0xff
mov rcx, 'A'

In [ ]:
mov rdx, rax      ; copy register to register

### Smaller parts of a register

Each 64-bit register has smaller names: `rax` (64 bits) → `eax` (low 32) → `ax` (low 16) → `al` (low 8).
Writing to `eax` clears the upper 32 bits; writing to `ax` or `al` doesn't.

In [ ]:
mov rax, -1       ; all 64 bits set: 0xffffffffffffffff
mov al, 0x12      ; only the lowest byte changes

In [ ]:
mov eax, 1        ; writing a 32-bit register zeroes the top half

## 3. Arithmetic

`add`, `sub`, `inc`, `dec`, `neg`, and `imul` for multiplication. The result goes into the first operand.

In [ ]:
mov rax, 10
add rax, 5        ; rax = 15
sub rax, 3        ; rax = 12
imul rax, rax, 2  ; rax = 24
inc rbx           ; rbx = 0x100

### Division

`div` / `idiv` divide the 128-bit number in `rdx:rax` by the operand: the quotient goes in `rax`, the remainder in `rdx`.
Set `rdx` to 0 first (or use `cqo` for signed numbers).

In [ ]:
mov rax, 17
xor rdx, rdx      ; rdx = 0 (xor with itself is the usual way to clear a register)
mov rcx, 5
div rcx           ; rax = 3, rdx = 2

### Bitwise operations

`and`, `or`, `xor`, `not`, and shifts `shl` / `shr` (multiply / divide by powers of 2).

In [ ]:
mov rax, 0b1100
and rax, 0b1010   ; 0b1000
or  rax, 0b0001   ; 0b1001
shl rax, 4        ; 0b10010000 = 144

## 4. Flags and comparisons

`cmp a, b` subtracts `b` from `a` **without storing the result**; it only sets the flags:

- **ZF** (zero flag) = 1 if they are equal
- **SF** (sign flag) = 1 if the result is negative
- **CF** (carry flag) = 1 if `a < b` treating them as unsigned
- **OF** (overflow flag) = 1 if the signed result overflowed

In [ ]:
mov rax, 5
cmp rax, 5        ; equal: ZF=1

In [ ]:
cmp rax, 9        ; 5 - 9 is negative: SF=1, and CF=1 (5 < 9)

In [ ]:
mov rbx, 0x7fffffffffffffff   ; the largest signed 64-bit number
add rbx, 1                    ; signed overflow: OF=1

## 5. Jumps and loops

A **label** (a name followed by `:`) marks a position in the code. `jmp label` always jumps;
conditional jumps look at the flags: `je`/`jz` (equal), `jne`/`jnz`, `jl`/`jg` (signed less/greater), `jb`/`ja` (unsigned)…

In this notebook a loop must be inside **one cell** (labels don't carry over between cells).

Sum of 1 + 2 + … + 10:

In [ ]:
    xor rax, rax      ; sum = 0
    mov rcx, 10       ; counter
again:
    add rax, rcx
    dec rcx           ; sets ZF when rcx reaches 0
    jnz again         ; loop while rcx != 0
    ; rax = 55

An *if / else* in assembly. Find the larger of `rbx` and `rdx`, and put it in `rax`:

In [ ]:
    mov rbx, 7
    mov rdx, 12
    mov rax, rbx      ; assume rbx is larger
    cmp rbx, rdx
    jge done          ; if rbx >= rdx we're done
    mov rax, rdx      ; else take rdx
done:

## 6. Constants

`equ` and `%define` give names to values. In this kernel they are remembered in later cells.

In [ ]:
LIMIT equ 100
%define STEP 7
mov rax, LIMIT

In [ ]:
sub rax, STEP

## 7. The stack: `push` and `pop`

The stack is memory that grows **downwards**. `push x` subtracts 8 from `rsp` and stores `x` there;
`pop x` loads the value at `rsp` into `x` and adds 8. Watch `rsp`:

In [ ]:
mov rax, 1111
mov rbx, 2222
push rax
push rbx

In [ ]:
%mem rsp 16

In [ ]:
pop rcx           ; last in, first out: rcx = 2222
pop rdx           ; rdx = 1111

## 8. Memory

This kernel gives you a scratch memory area called `DATA` (64 KiB). Square brackets mean *the memory at this address*:
`[DATA]` is the first byte, `[DATA + 8]` is 8 bytes later. When the size isn't clear, say it: `byte`, `word` (2), `dword` (4), `qword` (8).

x86 is **little-endian**: the lowest byte is stored first, so look carefully at the dump.

In [ ]:
mov qword [DATA], 0x11223344
mov byte [DATA + 8], 'H'
mov byte [DATA + 9], 'i'
mov rax, [DATA]   ; read 8 bytes back

In [ ]:
%mem DATA 16

### Arrays in memory

An array is consecutive values. Here we store 5 numbers (8 bytes each) and add them with a loop.
`[rsi + rcx*8]` is *base + index × size*.

In [ ]:
    mov rsi, DATA + 64        ; where our array starts
    mov qword [rsi], 3
    mov qword [rsi + 8], 1
    mov qword [rsi + 16], 4
    mov qword [rsi + 24], 1
    mov qword [rsi + 32], 5

    xor rax, rax              ; sum
    xor rcx, rcx              ; index
sum_loop:
    add rax, [rsi + rcx*8]
    inc rcx
    cmp rcx, 5
    jl sum_loop
    ; rax = 14

In [ ]:
%mem DATA+64 40

## 9. A complete program: Hello, world!

Real programs have **sections**: `.data` for initialised data, `.bss` for reserved space, and `.text` for code.
They start at the `_start` label and talk to the operating system with **system calls** (`syscall`):

| `rax` | System call | Arguments |
|---|---|---|
| 1 | `write` | `rdi` = file (1 = screen), `rsi` = address, `rdx` = length |
| 60 | `exit` | `rdi` = exit code |

Start the cell with `%%program` to run it as a whole program. This same code works on a real Linux machine:
`nasm -f elf64 hello.asm && ld hello.o -o hello && ./hello`.

In [ ]:
%%program
section .data
    msg     db  "Hello, world!", 10    ; 10 = newline
    len     equ $ - msg                ; $ = current address, so len = 14

section .text
global _start
_start:
    mov rax, 1          ; write(
    mov rdi, 1          ;   stdout,
    mov rsi, msg        ;   msg,
    mov rdx, len        ;   len)
    syscall

    mov rax, 60         ; exit(
    xor rdi, rdi        ;   0)
    syscall

## 10. Functions: `call` and `ret`

`call label` pushes the return address and jumps; `ret` pops it and jumps back. By convention, arguments go in
`rdi`, `rsi`, `rdx`… and the result comes back in `rax`.

This program prints a digit for each number from 1 to 5 using a helper function:

In [ ]:
%%program
section .bss
    char    resb 1

section .text
_start:
    mov rbx, 1              ; counter
.next:
    mov rdi, rbx
    call print_digit
    inc rbx
    cmp rbx, 5
    jle .next

    mov rax, 60
    mov rdi, 0
    syscall

; print_digit(rdi = number 0-9)
print_digit:
    add rdi, '0'            ; number -> ASCII character
    mov [char], dil         ; dil = lowest byte of rdi
    mov rax, 1
    mov rdi, 1
    mov rsi, char
    mov rdx, 1
    syscall
    ret

## Try it yourself

1. Compute 5! (5 × 4 × 3 × 2 × 1) in `rax` with a loop.
2. Store the numbers 10, 20, 30 in `DATA` and find the largest one.
3. Change the Hello world program to print your name, and exit with code 3.

In [ ]:
%reset

In [ ]:
; Your code here
mov rax, 1